# Hands-on Projects (Video Generation)

**Module:** 18 — Video Generation

Four projects: 4-second clip factory, storyboard→animatic, meeting highlight reel, provider bakeoff.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Ship scaffolds with acceptance criteria and smoke tests
- Practice job queues, storyboards, RAG highlights, and bakeoffs
- Document SLOs, costs, and failure modes


## Project map

```mermaid
flowchart LR
  P1[Clip Factory] --> P2[Storyboard Animatic]
  P2 --> P3[Meeting Highlights]
  P3 --> P4[Provider Bakeoff]
```


## Project 1 — 4-Second Clip Factory

**Brief:** Queue txt2video jobs at fixed 4s/24fps; cache by prompt+seed; expose poll API.

| ID | Acceptance |
|----|------------|
| A1 | Enqueue returns job_id |
| A2 | Identical prompt+seed hits cache |
| A3 | Status transitions queued→done |


In [ ]:
# Project 1 starter
from collections import deque
import hashlib, json

class ClipFactory:
    def __init__(self):
        self.q = deque(); self.jobs = {}; self.cache = {}
    def _key(self, prompt, seed):
        return hashlib.sha256(json.dumps({"p": prompt, "s": seed}, sort_keys=True).encode()).hexdigest()[:16]
    def enqueue(self, prompt, seed=0):
        key = self._key(prompt, seed)
        if key in self.cache:
            return self.cache[key]
        jid = f"job_{len(self.jobs)+1}"
        self.jobs[jid] = {"status": "queued", "prompt": prompt, "seed": seed, "url": None}
        self.q.append(jid); self.cache[key] = jid
        return jid
    def run_one(self):
        jid = self.q.popleft(); j = self.jobs[jid]
        j["status"] = "done"; j["url"] = f"https://example.invalid/{jid}.mp4"
        return j

cf = ClipFactory()
a = cf.enqueue("misty pines", 1); b = cf.enqueue("misty pines", 1)
assert a == b
print(cf.run_one())


### Try it yourself — Project 1

1. Add webhook list on done.
2. Reject prompts failing a safety stub.
3. Track cost_proxy per job.


## Project 2 — Storyboard to Animatic

**Brief:** Turn ordered still descriptions into img2video clip plans + timeline JSON.


In [ ]:
# Project 2 starter
def animatic(storyboard, seconds_per=3.0):
    clips = []
    t = 0.0
    for i, panel in enumerate(storyboard):
        clips.append({
            "panel": i,
            "image_prompt": panel,
            "video_prompt": panel + ", slow push-in",
            "track_in": t,
            "duration": seconds_per,
        })
        t += seconds_per
    return {"fps": 24, "clips": clips, "total": t}

print(animatic(["wide establishing town", "hero at door", "logo endcard"]))


## Project 3 — Meeting Highlight Reel

**Brief:** ASR moments → Video RAG → bounded timeline → optional TTS VO.


In [ ]:
# Project 3 starter
moments = [
    {"t0": 0, "t1": 4, "text": "agenda and goals"},
    {"t0": 30, "t1": 38, "text": "latency regression root cause"},
    {"t0": 90, "t1": 95, "text": "action items owners"},
]

def highlights(query, budget=15):
    q = set(query.split())
    ranked = sorted(moments, key=lambda m: -len(q & set(m["text"].split())))
    out, used = [], 0
    for m in ranked:
        dur = m["t1"]-m["t0"]
        if used + dur > budget: continue
        out.append(m); used += dur
    return out

print(highlights("latency action items"))


## Project 4 — Provider Bakeoff

**Brief:** Run the same 8 prompts across 2 mock providers; score motion/identity/cost.


In [ ]:
# Project 4 starter
prompts = [f"scene {i}" for i in range(8)]

def mock_score(provider, prompt):
    base = 0.7 + (hash(provider+prompt) % 20) / 100
    return {"adherence": base, "motion": base-0.05, "cost": 0.8 if provider=="open" else 0.5}

def bakeoff(providers):
    table = []
    for p in prompts:
        for prov in providers:
            s = mock_score(prov, p)
            table.append({"provider": prov, "prompt": p, **s, "sum": s["adherence"]+s["motion"]+s["cost"]})
    return sorted(table, key=lambda r: -r["sum"])[:5]

print(bakeoff(["sora_class", "open"]))


## Acceptance Checklist
- [ ] Async job lifecycle demonstrated
- [ ] Timeline JSON for animatic / highlights
- [ ] Safety stub on user prompts/uploads
- [ ] Cost column in bakeoff
- [ ] ≥5 smoke asserts


In [ ]:
# Shared smoke tests
assert cf.enqueue("misty pines", 1) == a
board = bakeoff(["sora_class", "open"])
assert board[0]["sum"] >= board[-1]["sum"]
assert animatic(["a", "b"])["total"] == 6.0
assert sum(m["t1"]-m["t0"] for m in highlights("latency", 15)) <= 15
print("smoke tests passed")


### Try it yourself — Ship

1. Persist ClipFactory jobs to a JSONL file.
2. Add beat-sync markers to Project 3 exports.

**Stretch:** Swap mocks for one real provider using YOUR_* API keys on 3 prompts.


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `animatic` | Timed storyboard preview approximating motion |
| `clip factory` | Service producing many short gens under policy |
| `bakeoff` | Cross-provider evaluation on a fixed suite |


### Workshop — Parameter journal — Video Projects

List every knob you touched. Predict the effect of changing one knob before changing it.


In [ ]:
# Workshop 1 — Video Projects
knobs = ['seed','guidance','steps','size','strength']
for k in knobs:
    print(f'{k}: value=?, hypothesis=?, observed=?')


### Workshop — Failure taxonomy — Video Projects

Classify bad outputs into: prompt, model, control, safety, or infra.


In [ ]:
# Workshop 2 — Video Projects
examples = ['ignored count','flicker','pose ignored','blocked','504']
for e in examples:
    print(e, '->', 'TODO-label')


### Workshop — Cost / latency card — Video Projects

Estimate unit cost for draft vs final tiers at a daily volume.


In [ ]:
# Workshop 3 — Video Projects
def monthly(qpd, price, days=30, retry=0.1):
    return round(qpd*days*(1+retry)*price, 2)
print('draft$', monthly(2000, 0.02))
print('final$', monthly(500, 0.08))


### Workshop — Eval golden item — Video Projects

Add one golden prompt/job with must-have attributes and reject criteria.


In [ ]:
# Workshop 4 — Video Projects
golden = {'id':'g1','prompt':'TODO','must_have':['subject','style'],'reject_if':['watermark']}
print(golden)


### Workshop — Safety + provenance — Video Projects

Write audit metadata: model hash, seed, policy version, credentials flag.


In [ ]:
# Workshop 5 — Video Projects
meta = {'model':'name@sha256:...','seed':0,'policy_version':'2026.04','credentials':True}
assert 'model' in meta
print(meta)


### Workshop — Ablation plan — Video Projects

Design a 4-run ablation changing only one variable each time.


In [ ]:
# Workshop 6 — Video Projects
runs = [{'id':i,'change':c,'score':None} for i,c in enumerate(['baseline','guidance-2','steps+10','new_seed'],1)]
print(runs)


## Key Takeaways

- Projects should exercise jobs, timelines, RAG, and bakeoffs
- Fixed durations and caches make factories operable
- Acceptance tests and cost columns keep demos honest
